In [32]:
# =============================================================================
# Projet DSTI – Prédiction des achats clients (Instacart)
# Code de départ propre – Chargement → Feature Engineering
# =============================================================================

In [33]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [34]:
print("=== Début du pipeline propre – Chargement & Préparation ===")
print("Date système :", pd.Timestamp.now().strftime("%Y-%m-%d"))

=== Début du pipeline propre – Chargement & Préparation ===
Date système : 2026-02-25


In [35]:
# -----------------------------------------------------------------------------
# 1. Chemins et paramètres
# -----------------------------------------------------------------------------
DATA_PATH = '../Data/'          # ← CHANGE SI NÉCESSAIRE
SAMPLE_USERS = 200000                # 20k users → rapide et représentatif
RANDOM_STATE = 42

In [36]:
# -----------------------------------------------------------------------------
# 2. Chargement des fichiers
# -----------------------------------------------------------------------------
print("\nChargement des fichiers Instacart...")

orders = pd.read_csv(DATA_PATH + 'orders.csv')
prior = pd.read_csv(DATA_PATH + 'order_products__prior.csv')
train = pd.read_csv(DATA_PATH + 'order_products__train.csv')
products = pd.read_csv(DATA_PATH + 'products.csv')

print(f"orders     : {orders.shape}")
print(f"prior      : {prior.shape}")
print(f"train      : {train.shape}")
print(f"products   : {products.shape}")


Chargement des fichiers Instacart...
orders     : (3421083, 7)
prior      : (32434489, 4)
train      : (1384617, 4)
products   : (49688, 4)


In [37]:
# -----------------------------------------------------------------------------
# 3. Échantillonnage (20 000 users aléatoires)
# -----------------------------------------------------------------------------
print("\nÉchantillonnage de", SAMPLE_USERS, "utilisateurs...")
sampled_users = orders['user_id'].sample(n=SAMPLE_USERS, random_state=RANDOM_STATE).unique()

orders_sample = orders[orders['user_id'].isin(sampled_users)].copy()
prior_sample  = prior[prior['order_id'].isin(orders_sample['order_id'])].copy()
train_sample  = train[train['order_id'].isin(orders_sample['order_id'])].copy()

print(f"orders_sample : {orders_sample.shape}")
print(f"prior_sample  : {prior_sample.shape}")
print(f"train_sample  : {train_sample.shape}")


Échantillonnage de 200000 utilisateurs...
orders_sample : (2474565, 7)
prior_sample  : (23961694, 4)
train_sample  : (727466, 4)


In [38]:
# -----------------------------------------------------------------------------
# 4. Création de prior_with_user (MERGE IMPORTANT – TOUTES COLONNES UTILES)
# -----------------------------------------------------------------------------
print("\nMerge prior + orders (ajout user_id + order_number + ...)")
prior_with_user = prior_sample.merge(
    orders_sample[['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']],
    on='order_id',
    how='left'
)

print("Colonnes dans prior_with_user :", prior_with_user.columns.tolist())
print("Vérification clé : 'order_number' présent ?", 'order_number' in prior_with_user.columns)


Merge prior + orders (ajout user_id + order_number + ...)
Colonnes dans prior_with_user : ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Vérification clé : 'order_number' présent ? True


In [39]:
# -----------------------------------------------------------------------------
# 5. Création de df_pred (user × product déjà achetés + label reorder)
# -----------------------------------------------------------------------------
print("\nCréation du dataset de prédiction (user-product pairs)...")

# Produits déjà achetés par utilisateur
user_products = prior_with_user.groupby('user_id')['product_id'].unique().reset_index()
user_products = user_products.explode('product_id')

# Labels = 1 si re-acheté dans le panier train
labels = train_sample[['order_id', 'product_id']].copy()
labels['label'] = 1
labels = labels.merge(orders_sample[['order_id', 'user_id']], on='order_id', how='left')

# Union left → label 0/1
df_pred = user_products.merge(labels[['user_id', 'product_id', 'label']], 
                              on=['user_id', 'product_id'], how='left')
df_pred['label'] = df_pred['label'].fillna(0).astype(int)

print(f"df_pred créé → {df_pred.shape[0]:,} lignes (user × produit déjà vu)")
print(df_pred.head(6))


Création du dataset de prédiction (user-product pairs)...
df_pred créé → 8,737,245 lignes (user × produit déjà vu)
   user_id product_id  label
0        2      49451      0
1        2      32792      1
2        2      32139      0
3        2      34688      0
4        2      36735      0
5        2      37646      0


In [40]:
# -----------------------------------------------------------------------------
# 6. FEATURE ENGINEERING COMPLET (partie 4 corrigée)
# -----------------------------------------------------------------------------

print("\n=== FEATURE ENGINEERING ===")

# 6.1 Features User
print("→ Features utilisateur...")
user_features_orders = orders_sample.groupby('user_id').agg(
    user_total_orders           = ('order_number', 'max'),
    user_avg_days_since_prior   = ('days_since_prior_order', 'mean'),
    user_median_days_since_prior = ('days_since_prior_order', 'median'),
    user_orders_dow_mode        = ('order_dow', lambda x: x.mode()[0] if not x.empty else -1),
    user_orders_hour_mode       = ('order_hour_of_day', lambda x: x.mode()[0] if not x.empty else -1)
).reset_index()

user_reorder_features = prior_with_user.groupby('user_id').agg(
    user_total_items           = ('product_id', 'count'),
    user_unique_products       = ('product_id', 'nunique'),
    user_total_orders_in_prior = ('order_id', 'nunique')
).reset_index()

user_reorder_features['user_reorder_rate'] = prior_with_user.groupby('user_id')['reordered'].mean().values
user_reorder_features['user_avg_basket_size'] = (
    user_reorder_features['user_total_items'] / 
    user_reorder_features['user_total_orders_in_prior'].replace(0, 1)
)

user_features = user_features_orders.merge(user_reorder_features, on='user_id', how='left')
user_features = user_features.fillna({
    'user_reorder_rate': 0.0,
    'user_avg_basket_size': 1.0,
    'user_total_items': 0,
    'user_unique_products': 0
})

# 6.2 Features Product
print("→ Features produit...")
product_features = prior_with_user.groupby('product_id').agg(
    product_total_purchases   = ('order_id', 'count'),
    product_unique_users      = ('user_id', 'nunique'),
    product_avg_add_to_cart   = ('add_to_cart_order', 'mean'),
    product_reorder_rate      = ('reordered', 'mean')
).reset_index()

# 6.3 Features User-Product
print("→ Features user-produit...")
user_product_features = prior_with_user.groupby(['user_id', 'product_id']).agg(
    user_product_orders         = ('order_id', 'nunique'),
    user_product_first_order    = ('order_number', 'min'),
    user_product_last_order     = ('order_number', 'max'),
    user_product_avg_cart_pos   = ('add_to_cart_order', 'mean'),
    user_product_reorder_rate   = ('reordered', 'mean')
).reset_index()

user_product_features['orders_since_last'] = (
    user_features.set_index('user_id')['user_total_orders']
    .reindex(user_product_features['user_id'])
    .values - user_product_features['user_product_last_order']
)

user_product_features['user_product_frequency'] = (
    user_product_features['user_product_orders'] / 
    user_features.set_index('user_id')['user_total_orders']
    .reindex(user_product_features['user_id'])
    .replace(0, 1)
    .values
)

# 6.4 Merge final
print("\nMerge final...")
df = df_pred.merge(user_features, on='user_id', how='left')
df = df.merge(product_features, on='product_id', how='left')
df = df.merge(user_product_features, on=['user_id', 'product_id'], how='left')

# Features supplémentaires
df['user_product_order_ratio'] = df['user_product_orders'] / df['user_total_orders'].replace(0, 1)
df['days_since_last_purchase'] = df['user_total_orders'] - df['user_product_last_order']

# Prix simulés
np.random.seed(42)
if 'price' not in products.columns:
    products['price'] = np.random.uniform(0.5, 15.0, size=len(products))
df = df.merge(products[['product_id', 'price']], on='product_id', how='left')

# Nettoyage NaN final
df = df.fillna({
    'user_reorder_rate': 0.0,
    'product_reorder_rate': 0.0,
    'user_product_reorder_rate': 0.0,
    'orders_since_last': 7.0,
    'user_product_frequency': 0.0,
    'price': 5.0
})

# -----------------------------------------------------------------------------
# Résultat
# -----------------------------------------------------------------------------
print("\n=== FIN DU PIPELINE PRÉPARATION ===")
print(f"Dataset final : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print("\nColonnes :")
print(list(df.columns))
print("\nAperçu :")
print(df.head(8))

print("\nValeurs manquantes restantes :", df.isna().sum().sum())
print("Prêt pour l'entraînement du modèle !")


=== FEATURE ENGINEERING ===
→ Features utilisateur...
→ Features produit...
→ Features user-produit...

Merge final...

=== FIN DU PIPELINE PRÉPARATION ===
Dataset final : 8,737,245 lignes × 27 colonnes

Colonnes :
['user_id', 'product_id', 'label', 'user_total_orders', 'user_avg_days_since_prior', 'user_median_days_since_prior', 'user_orders_dow_mode', 'user_orders_hour_mode', 'user_total_items', 'user_unique_products', 'user_total_orders_in_prior', 'user_reorder_rate', 'user_avg_basket_size', 'product_total_purchases', 'product_unique_users', 'product_avg_add_to_cart', 'product_reorder_rate', 'user_product_orders', 'user_product_first_order', 'user_product_last_order', 'user_product_avg_cart_pos', 'user_product_reorder_rate', 'orders_since_last', 'user_product_frequency', 'user_product_order_ratio', 'days_since_last_purchase', 'price']

Aperçu :
   user_id product_id  label  user_total_orders  user_avg_days_since_prior  \
0        2      49451      0                 15              

In [41]:
# =============================================================================
# 7. ENTRAÎNEMENT DU MODÈLE – Prédiction de réachat (LightGBM)
# =============================================================================

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

print("\n=== ENTRAÎNEMENT DU MODÈLE ===")

# Sélection des features (exclure les ID et la cible)
exclude_cols = ['user_id', 'product_id', 'label']
features = [col for col in df.columns if col not in exclude_cols]

X = df[features]
y = df['label']

# Split train / validation (stratifié car classes déséquilibrées)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"X_train : {X_train.shape}, y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape},   y_val   : {y_val.shape}")

# Dataset LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Paramètres (bons par défaut pour Instacart)
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42,
    'is_unbalance': True   # aide avec le déséquilibre
}

print("Entraînement en cours...")
model = lgb.train(
    params,
    train_data,
    num_boost_round=1500,
    valid_sets=[val_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=80),
        lgb.log_evaluation(period=100)
    ]
)


=== ENTRAÎNEMENT DU MODÈLE ===
X_train : (6989796, 24), y_train : (6989796,)
X_val   : (1747449, 24),   y_val   : (1747449,)
Entraînement en cours...
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.200988


In [42]:
# Prédictions probabilités sur validation
val_proba = model.predict(X_val)

In [43]:
# Optimisation du seuil pour maximiser F1
print("\nOptimisation du seuil de décision...")
thresholds = np.arange(0.05, 0.51, 0.025)
best_f1, best_threshold = 0, 0.25

for thresh in thresholds:
    pred_binary = (val_proba > thresh).astype(int)
    current_f1 = f1_score(y_val, pred_binary)
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = thresh

print(f"Meilleur seuil trouvé : {best_threshold:.3f}")
print(f"F1-score validation   : {best_f1:.4f}")

pred_optimal = (val_proba > best_threshold).astype(int)
print(f"Precision : {precision_score(y_val, pred_optimal):.4f}")
print(f"Recall    : {recall_score(y_val, pred_optimal):.4f}")


Optimisation du seuil de décision...
Meilleur seuil trouvé : 0.100
F1-score validation   : 0.3123
Precision : 0.2365
Recall    : 0.4599


In [44]:
# Optimisation du seuil pour maximiser F1
print("\nOptimisation du seuil de décision...")
thresholds = np.arange(0.05, 0.51, 0.025)
best_f1, best_threshold = 0, 0.25

for thresh in thresholds:
    pred_binary = (val_proba > thresh).astype(int)
    current_f1 = f1_score(y_val, pred_binary)
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = thresh

print(f"Meilleur seuil trouvé : {best_threshold:.3f}")
print(f"F1-score validation   : {best_f1:.4f}")

pred_optimal = (val_proba > best_threshold).astype(int)
print(f"Precision : {precision_score(y_val, pred_optimal):.4f}")
print(f"Recall    : {recall_score(y_val, pred_optimal):.4f}")


Optimisation du seuil de décision...
Meilleur seuil trouvé : 0.100
F1-score validation   : 0.3123
Precision : 0.2365
Recall    : 0.4599


In [45]:
# Sauvegarde du modèle
model.save_model('lightgbm_reorder_model.txt')
print("\nModèle sauvegardé : lightgbm_reorder_model.txt")


Modèle sauvegardé : lightgbm_reorder_model.txt


In [48]:
# Exemple : prendre un user_id qui a assez de produits
user_example = df['user_id'].value_counts().index[4]  # le 5ème plus fréquent par exemple

user_data = df[df['user_id'] == user_example].copy()

# Ajouter les probabilités (si pas déjà fait)
user_data['proba'] = model.predict(user_data[features])

# Trier par probabilité descendante
top_pred = user_data.sort_values('proba', ascending=False).head(10)

print(f"Top 10 produits prédits pour user_id {user_example} :")
print(top_pred[['product_id', 'proba', 'label', 'price', 'user_product_orders']])

# Comparer avec la réalité (label)
print("\nProduits réellement ré-achetés (label=1) :")
print(top_pred[top_pred['label'] == 1][['product_id', 'proba', 'price']])

Top 10 produits prédits pour user_id 116571 :
        product_id     proba  label      price  user_product_orders
4954030      20259  0.110062      0   8.817284                   24
4954503      27275  0.105349      0   4.939227                    3
4953988      47242  0.105349      0   6.166447                    7
4954045      22882  0.105349      0  10.459041                   21
4954057      31964  0.105349      0   7.257841                   11
4954073       2639  0.105349      0  14.286204                    7
4954074      22021  0.105349      0  11.296224                    8
4954133       5550  0.105349      0   2.467091                   13
4954072      42372  0.105349      0   6.521595                    6
4953993      16797  0.101746      0   2.579570                   23

Produits réellement ré-achetés (label=1) :
Empty DataFrame
Columns: [product_id, proba, price]
Index: []


In [49]:
# Exemple : prendre un user_id qui a assez de produits
user_example = df['user_id'].value_counts().index[10]  # le 5ème plus fréquent par exemple

user_data = df[df['user_id'] == user_example].copy()

# Ajouter les probabilités (si pas déjà fait)
user_data['proba'] = model.predict(user_data[features])

# Trier par probabilité descendante
top_pred = user_data.sort_values('proba', ascending=False).head(10)

print(f"Top 10 produits prédits pour user_id {user_example} :")
print(top_pred[['product_id', 'proba', 'label', 'price', 'user_product_orders']])

# Comparer avec la réalité (label)
print("\nProduits réellement ré-achetés (label=1) :")
print(top_pred[top_pred['label'] == 1][['product_id', 'proba', 'price']])

Top 10 produits prédits pour user_id 290 :
      product_id     proba  label      price  user_product_orders
12143      30251  0.116851      0   7.577531                   28
12180      42895  0.116851      0  13.344696                   33
12124      23405  0.116851      0   8.001350                   46
12121      34355  0.116851      0   9.008024                   45
12181      35759  0.110062      0   8.327730                   17
12134      16096  0.110062      0  14.707146                   17
12327      46541  0.110062      0   1.839726                   16
12173      25810  0.110062      0  13.345014                   16
12304      37980  0.105349      0   7.272772                    8
12182      12902  0.105349      0   4.436521                    8

Produits réellement ré-achetés (label=1) :
Empty DataFrame
Columns: [product_id, proba, price]
Index: []


In [50]:
# Trouver un utilisateur qui a au moins 3 vrais réachats dans le set train
users_with_reorder = df[df['label'] == 1]['user_id'].value_counts()
user_example = users_with_reorder[users_with_reorder >= 3].index[0]  # premier avec >=3 réachats

print(f"Utilisateur choisi : {user_example} (qui a {users_with_reorder[user_example]} vrais réachats)")

user_data = df[df['user_id'] == user_example].copy()
user_data['proba'] = model.predict(user_data[features])

top_pred = user_data.sort_values('proba', ascending=False).head(15)  # 15 au lieu de 10

print(f"\nTop 15 produits prédits pour user_id {user_example} :")
print(top_pred[['product_id', 'proba', 'label', 'price', 'user_product_orders']])

print("\nParmi ces top 15, les vrais ré-achetés (label=1) :")
real_reorders = top_pred[top_pred['label'] == 1]
print(real_reorders[['product_id', 'proba', 'price']])

if real_reorders.empty:
    print("Toujours vide → cet utilisateur a très peu de réachats ou le modèle est trop conservateur")
else:
    print(f"→ Le modèle a capturé {len(real_reorders)} vrais réachats dans son top 15")

Utilisateur choisi : 181991 (qui a 71 vrais réachats)

Top 15 produits prédits pour user_id 181991 :
        product_id     proba  label      price  user_product_orders
7707167      13740  0.116851      1  13.131970                   26
7707022      19987  0.116851      0   6.674349                   26
7707021      30337  0.116851      0   1.655649                   26
7707020       7781  0.116851      1   7.104775                   31
7707135      28920  0.116851      1   9.400157                   31
7707103      29285  0.116851      1   3.132078                   26
7707101      43428  0.116851      1  13.224248                   31
7707080       5322  0.116851      1   4.545620                   28
7707068      20785  0.116851      1  10.488950                   27
7707065      19019  0.116851      1   6.582761                   34
7707064      27344  0.116851      1  13.858483                   35
7707051      10106  0.116851      1   3.727753                   26
7707037      45